### 1. Ternary STE Microbenchmark

In [ ]:
import torch
import torch.nn.functional as F
import time

torch.manual_seed(42)
device = torch.device('cuda')

# -----------------------------------------------------------------------
# Match real model dimensions -- largest ternary matrix is MLP fc = [1536, 768].
# Test on this as it's the biggest layer.
# -----------------------------------------------------------------------

W          = torch.randn(1536, 768, device=device, dtype=torch.bfloat16)
GROUP_SIZE = 128
N_WARMUP   = 200
N_RUNS     = 2000


def ste_current(w):
    g     = GROUP_SIZE
    w_g   = w.reshape(-1, g)
    scale = w_g.abs().mean(-1, keepdim=True).clamp(min=1e-8)
    q     = (w_g / scale).round().clamp(-1, 1)
    return w + ((q * scale).reshape(w.shape) - w).detach()


def ste_optimized(w):
    g     = GROUP_SIZE
    w_g   = w.reshape(-1, g)
    scale = w_g.abs().mean(-1, keepdim=True).clamp(min=1e-8)
    q     = (w_g / scale).round().clamp_(-1, 1)
    return w + ((q * scale).view_as(w) - w).detach()


def ste_v2(w):
    g         = GROUP_SIZE
    w_g       = w.reshape(-1, g)
    scale     = w_g.abs().mean(-1, keepdim=True).clamp(min=1e-8)
    inv_scale = scale.reciprocal()
    q         = (w_g * inv_scale).round().clamp(-1, 1)
    return w + ((q * scale).view_as(w) - w).detach()


for _ in range(N_WARMUP):
    _ = ste_current(W)
    _ = ste_optimized(W)
    _ = ste_v2(W)
torch.cuda.synchronize()

for name, fn in [
    ('Current STE',       ste_current),
    ('Optimized STE',     ste_optimized),
    ('STE v2 (recip)',    ste_v2),
]:
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(N_RUNS):
        _ = fn(W)
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - t0) / N_RUNS * 1000
    print(f'{name:<25} {elapsed:.4f} ms/call')

# Per block: c_qkv(1) + attn_proj(1) + mlp_fc(1) + mlp_proj(1) = 4.
# 12 blocks = 48 STE calls per step.
print(f'\n48 STE calls/step (12L): current = {48 * elapsed:.2f} ms overhead estimate')

### 2. Contiguous() Necessity Check

In [ ]:
import torch
import torch.nn.functional as F
import time

device = torch.device('cuda')

# -----------------------------------------------------------------------
# Simulate the exact path in CausalSelfAttention.forward after fused QKV split.
# Tests whether q / k / v are contiguous before flash_attn is called.
# -----------------------------------------------------------------------

B, S, H, KV_H, D = 64, 1024, 8, 4, 96
q_size  = H    * D
kv_size = KV_H * D
qkv     = torch.randn(B, S, q_size + 2 * kv_size, device=device, dtype=torch.bfloat16)

q_out, k_out, v_out = qkv.split([q_size, kv_size, kv_size], dim=-1)
q = q_out.reshape(B, S, H,    D)
k = k_out.reshape(B, S, KV_H, D)
v = v_out.reshape(B, S, KV_H, D)

print('After reshape (pre-RoPE):')
print(f'  q contiguous: {q.is_contiguous()}, stride: {q.stride()}')
print(f'  k contiguous: {k.is_contiguous()}, stride: {k.stride()}')
print(f'  v contiguous: {v.is_contiguous()}, stride: {v.stride()}')

q = F.rms_norm(q, (D,))
k = F.rms_norm(k, (D,))

half = D // 2
cos  = torch.randn(1, S, 1, half, device=device, dtype=torch.bfloat16)
sin  = torch.randn(1, S, 1, half, device=device, dtype=torch.bfloat16)
q1, q2 = q[..., :half], q[..., half:]
q      = torch.cat((q1 * cos + q2 * sin, q1 * (-sin) + q2 * cos), dim=-1)
k1, k2 = k[..., :half], k[..., half:]
k      = torch.cat((k1 * cos + k2 * sin, k1 * (-sin) + k2 * cos), dim=-1)
q      = q * 2.25

print('\nAfter RoPE + scale (pre flash_attn):')
print(f'  q contiguous: {q.is_contiguous()}, stride: {q.stride()}')
print(f'  k contiguous: {k.is_contiguous()}, stride: {k.stride()}')
print(f'  v contiguous: {v.is_contiguous()}, stride: {v.stride()}')

N_WARMUP, N_RUNS = 200, 2000

if q.is_contiguous() and k.is_contiguous() and v.is_contiguous():
    print('\nAll tensors already contiguous -- .contiguous() is a no-op.')
    print('Removing them saves only Python-level overhead (~0.01ms total).')
else:
    for name, tensor in [('q', q), ('k', k), ('v', v)]:
        if not tensor.is_contiguous():
            for _ in range(N_WARMUP):
                _ = tensor.contiguous()
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(N_RUNS):
                _ = tensor.contiguous()
            torch.cuda.synchronize()
            ms = (time.perf_counter() - t0) / N_RUNS * 1000
            print(f'\n  {name}.contiguous(): {ms:.4f} ms/call')
            print(f'  12L cost: {12 * ms:.3f} ms/step')
        else:
            print(f'\n  {name} already contiguous -- no cost')

### 3. RoPE Implementation Comparison

In [ ]:
import torch
import time

device        = torch.device('cuda')
B, S, H, D    = 64, 1024, 8, 96
half          = D // 2
N_WARMUP, N_RUNS = 200, 2000

q   = torch.randn(B, S, H, D, device=device, dtype=torch.bfloat16)
cos = torch.randn(1, S, 1, half, device=device, dtype=torch.bfloat16)
sin = torch.randn(1, S, 1, half, device=device, dtype=torch.bfloat16)


def rope_cat(x, cos, sin):
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat((x1 * cos + x2 * sin, x1 * (-sin) + x2 * cos), dim=-1)


def rope_stack(x, cos, sin):
    x1, x2 = x[..., :half], x[..., half:]
    r1     = x1 * cos + x2 * sin
    r2     = x1 * (-sin) + x2 * cos
    return torch.stack((r1, r2), dim=-1).reshape_as(x)


out_buf = torch.empty_like(q)
def rope_inplace(x, cos, sin):
    x1, x2 = x[..., :half], x[..., half:]
    out_buf[..., :half] = x1 * cos + x2 * sin
    out_buf[..., half:] = x1 * (-sin) + x2 * cos
    return out_buf


cos_f32 = torch.randn(1, S, 1, half, device=device, dtype=torch.float32)
sin_f32 = torch.randn(1, S, 1, half, device=device, dtype=torch.float32)
def rope_complex(x, cos, sin):
    x_f = x.float()
    x1, x2 = x_f[..., :half], x_f[..., half:]
    r1  = x1 * cos - x2 * sin
    r2  = x1 * sin + x2 * cos
    return torch.cat((r1, r2), dim=-1).to(x.dtype)


for name, fn, c, s in [
    ('Current (cat)',   rope_cat,     cos,     sin),
    ('Stack+reshape',   rope_stack,   cos,     sin),
    ('In-place buf',    rope_inplace, cos,     sin),
    ('Complex (fp32)',  rope_complex, cos_f32, sin_f32),
]:
    for _ in range(N_WARMUP):
        _ = fn(q, c, s)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(N_RUNS):
        _ = fn(q, c, s)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / N_RUNS * 1000
    print(f'{name:<20} {ms:.4f} ms/call   24 calls/step: {24 * ms:.3f} ms')

### 4. Softcap Polynomial vs Alternatives

In [ ]:
import torch
import time

device        = torch.device('cuda')
N_WARMUP, N_RUNS = 200, 2000

# -----------------------------------------------------------------------
# Logits shape: [batch*seq, vocab] = [64*1024, 8192] = [65536, 8192].
# -----------------------------------------------------------------------

logits = torch.randn(65536, 8192, device=device, dtype=torch.bfloat16)
S      = 10.0


def softcap_poly5(logits):
    x_sc = torch.clamp(logits / S, -2.0, 2.0)
    x2   = x_sc * x_sc
    return S * torch.clamp(x_sc * (1.0 - x2 / 3.0 + x2 * x2 / 15.0), -1.0, 1.0)


def softcap_poly3(logits):
    x_sc = torch.clamp(logits / S, -2.0, 2.0)
    return S * torch.clamp(x_sc * (1.0 - x_sc * x_sc / 3.0), -1.0, 1.0)


def softcap_hardtanh(logits):
    return torch.clamp(logits, -S, S)


def softcap_tanh(logits):
    return S * torch.tanh(logits / S)


test = torch.linspace(-25, 25, 1000, device=device)
ref  = S * torch.tanh(test / S)
for name, fn in [('Poly5', softcap_poly5), ('Poly3', softcap_poly3), ('Hardtanh', softcap_hardtanh)]:
    approx  = fn(test)
    max_err = (approx - ref).abs().max().item()
    print(f'{name:<12} max_err vs tanh: {max_err:.4f}')

print()

for name, fn in [
    ('Current poly5', softcap_poly5),
    ('Poly3',         softcap_poly3),
    ('Hardtanh',      softcap_hardtanh),
    ('Tanh',          softcap_tanh),
]:
    for _ in range(N_WARMUP):
        _ = fn(logits)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(N_RUNS):
        _ = fn(logits)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / N_RUNS * 1000
    print(f'{name:<18} {ms:.4f} ms/call')

### 5. Cross-entropy and Z-loss Fusion

In [ ]:
import torch
import torch.nn.functional as F
import time

device        = torch.device('cuda')
N_WARMUP, N_RUNS = 200, 2000

# -----------------------------------------------------------------------
# Current path: separate logsumexp + cross_entropy (both compute LSE).
# Fused path: extract LSE from manual CE to avoid double compute.
# -----------------------------------------------------------------------

logits  = torch.randn(65536, 8192, device=device, dtype=torch.float32)
targets = torch.randint(0, 8192, (65536,), device=device)


def loss_current(logits, targets):
    lse    = torch.logsumexp(logits, dim=-1)
    z_loss = 1e-4 * (lse ** 2).mean()
    ce     = F.cross_entropy(logits, targets, reduction='mean')
    return ce + z_loss


def loss_fused(logits, targets):
    lse           = torch.logsumexp(logits, dim=-1)
    target_logits = logits.gather(1, targets.unsqueeze(1)).squeeze(1)
    ce            = (lse - target_logits).mean()
    z_loss        = 1e-4 * (lse ** 2).mean()
    return ce + z_loss


l1 = loss_current(logits, targets)
l2 = loss_fused(logits, targets)
print(f'Current : {l1.item():.6f}')
print(f'Fused   : {l2.item():.6f}')
print(f'Diff    : {abs(l1.item() - l2.item()):.8f}')
print()

for name, fn in [('Current (separate)', loss_current), ('Fused (shared LSE)', loss_fused)]:
    for _ in range(N_WARMUP):
        _ = fn(logits, targets)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(N_RUNS):
        _ = fn(logits, targets)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / N_RUNS * 1000
    print(f'{name:<25} {ms:.4f} ms/call')

print('\nNote: savings are per-step (1 call). Also check backward pass cost.')

print('\nWith backward pass:')
for name, fn in [('Current (separate)', loss_current), ('Fused (shared LSE)', loss_fused)]:
    logits_p = logits.clone().requires_grad_(True)
    for _ in range(N_WARMUP):
        l = fn(logits_p, targets)
        l.backward()
        logits_p.grad = None
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(N_RUNS):
        l = fn(logits_p, targets)
        l.backward()
        logits_p.grad = None
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / N_RUNS * 1000
    print(f'{name:<25} {ms:.4f} ms/call (fwd+bwd)')